# 03 — Evaluate a Dataset File

This notebook shows how to:

1. Load evaluation cases from `.jsonl` or `.json`
2. Configure metrics with `build_metrics()`
3. Run the evaluator and get a `RunReport`
4. Inspect per-sample pass/fail, metric pass rates, and cost
5. Write the report to a JSON or Markdown file

Uses a `ReplayJudge` so no API key is needed.

In [12]:
from pprint import pprint

from evalf import Evaluator
from evalf.inputs import load_cases_from_path
from evalf.llms.base import BaseLLMModel
from evalf.metrics import build_metrics
from evalf.metrics.answer_correctness.schema import AnswerCorrectnessAssessment
from evalf.metrics.decomposition import (
    Claim,
    ClaimExtraction,
    ClaimSupportAssessment,
    ClaimSupportVerdict,
)
from evalf.reporting import report_to_json, report_to_markdown, write_report
from evalf.schemas import LLMResponse, UsageStats

## 1. Load cases from a JSONL file

In [2]:
cases = load_cases_from_path("../examples/rag_eval.jsonl")

print(f"Loaded {len(cases)} case(s)")
for c in cases:
    print(f"  {c.id}: {c.question[:60]}...")

Loaded 2 case(s)
  case-1: Under FERPA, when do rights transfer from parents to a stude...
  case-2: Under FERPA, when do rights transfer from parents to a stude...


## 2. Build metrics with `build_metrics()`

This is the same factory the CLI uses. You can set:
- a default threshold for all metrics
- per-metric threshold overrides
- `mode` (`pass@k` / `pass^k`) and `k`

In [3]:
metrics = build_metrics(
    ["faithfulness", "answer_correctness"],
    default_threshold=0.7,
    threshold_overrides={"answer_correctness": 0.8},
)

for m in metrics:
    print(f"  {m.name}: threshold={m.threshold}, mode={m.mode}, k={m.k}")

  faithfulness: threshold=0.7, mode=pass@k, k=1
  answer_correctness: threshold=0.8, mode=pass@k, k=1


## 3. ReplayJudge

In [4]:
class ReplayJudge(BaseLLMModel):
    def __init__(self, outputs):
        super().__init__(
            provider="test", model="replay", base_url="https://example.com", api_key=None
        )
        self._outputs = list(outputs)

    def _pop(self):
        if not self._outputs:
            raise AssertionError("Replay queue exhausted.")
        return LLMResponse(
            text=None,
            model=self.model,
            provider=self.provider,
            parsed_output=self._outputs.pop(0),
            usage=UsageStats(
                input_tokens=100,
                output_tokens=50,
                total_tokens=150,
                latency_ms=200.0,
                cost_usd=0.00015,
            ),
        )

    def generate(self, **kw):
        return self._pop()

    async def a_generate(self, **kw):
        return self._pop()

In [5]:
replay_outputs = [
    # --- case-1: faithfulness (extract + verify) ---
    ClaimExtraction(
        claims=[
            Claim(claim_id="c1", text="Rights transfer at age 18."),
            Claim(claim_id="c2", text="Rights transfer when entering postsecondary."),
        ]
    ),
    ClaimSupportAssessment(
        verdicts=[
            ClaimSupportVerdict(
                claim_id="c1",
                verdict="supported",
                evidence_context_ids=["ctx_1"],
                reason="Supported.",
            ),
            ClaimSupportVerdict(
                claim_id="c2",
                verdict="supported",
                evidence_context_ids=["ctx_1"],
                reason="Supported.",
            ),
        ]
    ),
    # --- case-1: answer_correctness ---
    AnswerCorrectnessAssessment(score=0.9, reason="Both conditions covered."),
    # --- case-2: faithfulness (extract + verify) ---
    ClaimExtraction(
        claims=[
            Claim(claim_id="c1", text="Rights transfer at age 21."),
        ]
    ),
    ClaimSupportAssessment(
        verdicts=[
            ClaimSupportVerdict(
                claim_id="c1",
                verdict="contradicted",
                evidence_context_ids=["ctx_1"],
                reason="Context says 18, not 21.",
            ),
        ]
    ),
    # --- case-2: answer_correctness ---
    AnswerCorrectnessAssessment(score=0.0, reason="Wrong age and missing second condition."),
]

## 4. Run evaluation

In [6]:
report = await Evaluator(judge=ReplayJudge(replay_outputs), concurrency=1).a_evaluate(
    cases=cases,
    metrics=metrics,
)

## 5. Inspect results

In [7]:
print("=== Run Summary ===")
pprint(report.summary.model_dump(exclude_none=True))

=== Run Summary ===
{'avg_latency_ms_per_sample': 600.0,
 'failed_samples': 1,
 'metric_pass_rates': {'answer_correctness': 0.5, 'faithfulness': 0.5},
 'passed_samples': 1,
 'skipped_samples': 0,
 'total_cost_usd': 0.0009,
 'total_input_tokens': 600,
 'total_output_tokens': 300,
 'total_samples': 2,
 'total_tokens': 900}


In [8]:
print("=== Per-sample results ===")
for sample in report.samples:
    print(f"\n{sample.sample_id}: {sample.status}")
    for m in sample.metrics:
        print(f"  {m.name:25s}  score={m.score}  status={m.status}  reason={m.reason}")

=== Per-sample results ===

case-1: passed
  faithfulness               score=1.0  status=passed  reason=Passed in 1/1 evaluated attempt(s) under pass@k.
  answer_correctness         score=0.9  status=passed  reason=Passed in 1/1 evaluated attempt(s) under pass@k.

case-2: failed
  faithfulness               score=0.0  status=failed  reason=No attempt met the threshold across 1 evaluated attempt(s) under pass@k.
  answer_correctness         score=0.0  status=failed  reason=No attempt met the threshold across 1 evaluated attempt(s) under pass@k.


## 6. Export the report

`write_report()` supports `.json` and `.md` extensions.
Extensionless paths default to `.json`.

In [9]:
# Preview the JSON output
print(report_to_json(report)[:500], "...")

{
  "run_id": "run_00af9580fbb1",
  "summary": {
    "total_samples": 2,
    "passed_samples": 1,
    "failed_samples": 1,
    "skipped_samples": 0,
    "total_input_tokens": 600,
    "total_output_tokens": 300,
    "total_tokens": 900,
    "total_cost_usd": 0.0009,
    "avg_latency_ms_per_sample": 600.0,
    "metric_pass_rates": {
      "answer_correctness": 0.5,
      "faithfulness": 0.5
    }
  },
  "samples": [
    {
      "sample_id": "case-1",
      "status": "passed",
      "metrics": [
  ...


In [10]:
# Preview the Markdown output
print(report_to_markdown(report)[:600], "...")

# evalf Report

- Run ID: `run_00af9580fbb1`
- Total Samples: `2`
- Passed Samples: `1`
- Failed Samples: `1`
- Skipped Samples: `0`
- Total Tokens: `900`
- Total Cost (USD): `0.0009`

## Samples

### case-1
- Status: `passed`
- Total Tokens: `450`
- Total Cost (USD): `0.00045`
- `faithfulness`: status=`passed`, mode=`pass@k`, k=`1`, score=`1.0`, threshold=`0.7`, cost_usd=`0.0003`
- `answer_correctness`: status=`passed`, mode=`pass@k`, k=`1`, score=`0.9`, threshold=`0.8`, cost_usd=`0.00015`

### case-2
- Status: `failed`
- Total Tokens: `450`
- Total Cost (USD): `0.00045`
- `faithfulness`: sta ...


In [13]:
# Write to disk (uncomment to actually write)
write_report(report, "/tmp/evalf_demo_report.json")
write_report(report, "/tmp/evalf_demo_report.md")

PosixPath('/tmp/evalf_demo_report.md')